# LiDAR Gap Analysis — Puerto Rico G-LiHT (Full Area)
**Luquillo Experimental Forest** | Gap creation: March 2017 (pre-Maria) → May 2018 (post-Maria)

Pipeline:
1. Load every file in each folder, subsample per file, merge into two full-area clouds
2. Find actual point-coverage overlap via 2D voxel grid — clip both clouds to shared area
3. C2C once across the whole merged area
4. Connected components + power-law gap size analysis

In [ ]:
import os, re, csv, sys, time, gzip, signal, shutil, tempfile, threading
from math import log2, sqrt
from pathlib import Path

import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

import cloudComPy as cc
cc.initCC()
print('[INIT] CloudComPy initialized')

## 1. Configuration

In [ ]:
# ── Input folders ─────────────────────────────────────────────────────────────
DIR_2017 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_11March2017_FIA12/lidar/las')
DIR_2018 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_2May018_FIA12/lidar/las')

# ── Output folders ────────────────────────────────────────────────────────────
OUT_DIR  = Path('/home/ogh6/Downloads/raw_c2c_output')
CSV_DIR  = OUT_DIR / 'csv'
PLOT_DIR = OUT_DIR / 'plots'
BIN_DIR  = OUT_DIR / 'bin'
for d in [OUT_DIR, CSV_DIR, PLOT_DIR, BIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Subsampling ───────────────────────────────────────────────────────────────
# Set to None to load every point (raw). Set to an integer to subsample per file.
SUBSAMPLE_N_PER_FILE = None

# ── Per-file load timeout ─────────────────────────────────────────────────────
FILE_LOAD_TIMEOUT_S  = 120

# ── Processing ────────────────────────────────────────────────────────────────
THREADS              = 12
NOISE_CEIL_M         = 20.0
C2C_THRESHOLDS       = [2, 5, 10, 15]
# Fixed threshold for "extreme outlier" reporting — points above this are
# likely noise, gross misregistration, or very large tree falls
EXTREME_THRESHOLD_M  = 25.0
HIST_MAX_M           = 20.0
HIST_BINS            = 200

# ── Overlap detection ─────────────────────────────────────────────────────────
OVERLAP_VOXEL_M  = 50.0
OVERLAP_BUFFER_M = 100.0
MIN_OVERLAP_FRAC = 0.30

# ── Connected components ──────────────────────────────────────────────────────
CC_CONFIG = {
    'damage_threshold':  2.0,
    'min_component_pts': 100,
    'min_gap_area_m2':   10.0,
    'max_components':    500_000,
}

_TMP_DIR = Path(tempfile.gettempdir()) / f'cc_c2c_{int(time.time())}'
_TMP_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  2017 : {DIR_2017}')
print(f'  2018 : {DIR_2018}')
print(f'  Out  : {OUT_DIR}')
print(f'  Subsampling      : {"disabled (raw)" if not SUBSAMPLE_N_PER_FILE else f"{SUBSAMPLE_N_PER_FILE:,} pts/file"}')
print(f'  Per-file timeout : {FILE_LOAD_TIMEOUT_S}s')
print(f'  Extreme threshold: >={EXTREME_THRESHOLD_M}m')

## 2. Utility Functions

In [ ]:
def tnow(): return time.time()
def fmt_s(dt):
    if dt < 60:   return f'{dt:.1f}s'
    if dt < 3600: return f'{dt/60:.1f}m'
    return f'{dt/3600:.2f}h'

def mem_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except ImportError: return -1.0

def gunzip_if_needed(p):
    p = Path(p)
    if p.suffix != '.gz': return p
    out = _TMP_DIR / p.name[:-3]
    if out.exists() and out.stat().st_size > 0: return out
    print(f'  [UNZIP] {p.name} ...', flush=True)
    with gzip.open(p, 'rb') as fi, open(out, 'wb') as fo: shutil.copyfileobj(fi, fo)
    return out

def safe_delete(entity):
    try:
        if entity is not None: cc.deleteEntity(entity)
    except Exception: pass

def write_csv(filepath, rows, fieldnames=None):
    if not rows: return
    if fieldnames is None:
        seen = {}
        for r in rows:
            for k in r: seen[k] = True
        fieldnames = list(seen)
    with open(filepath, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    print(f'  [CSV] {filepath}', flush=True)

print('Utility functions ready.')

## 3. Load & Merge Folder
Loads every `.las`/`.las.gz` file in a folder, subsamples each to `SUBSAMPLE_N_PER_FILE`, then merges all into one cloud using `cc.MergeEntities`. The per-file subsampling keeps memory bounded regardless of how many tiles there are.

In [ ]:
def subsample_cloud(cloud, target_n):
    if cloud.size() <= target_n: return cloud
    ref = cc.CloudSamplingTools.subsampleCloudWithOctree(
        cloud, int(target_n), cc.SUBSAMPLING_CELL_METHOD.NEAREST_POINT_TO_CELL_CENTER)
    sub, res = cloud.partialClone(ref)
    if res != 0: raise RuntimeError(f'partialClone failed (code {res})')
    return sub

# ──────────────────────────────────────────────────────────────────────────────
# Helpers for per-file timeout + heartbeat
# ──────────────────────────────────────────────────────────────────────────────

class _TimeoutError(Exception): pass

def _sigalrm_handler(sig, frame):
    raise _TimeoutError()

def _load_with_timeout(path_str, global_shift, timeout_s):
    """
    Load a LAS file with an optional SIGALRM timeout.
    Returns: cloud object | None (load failed) | 'TIMEOUT'
    SIGALRM is only available on Linux/Mac (not Windows).
    """
    def _do_load():
        if global_shift is not None:
            return cc.loadPointCloud(
                filename=path_str, mode=cc.CC_SHIFT_MODE.XYZ,
                x=global_shift[0], y=global_shift[1], z=global_shift[2])
        else:
            return cc.loadPointCloud(path_str)

    if timeout_s and timeout_s > 0 and hasattr(signal, 'SIGALRM'):
        old_handler = signal.signal(signal.SIGALRM, _sigalrm_handler)
        signal.alarm(int(timeout_s))
        try:
            return _do_load()
        except _TimeoutError:
            return 'TIMEOUT'
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old_handler)
    else:
        return _do_load()

def _heartbeat(file_name, stop_evt, interval_s=30):
    """Background thread: print elapsed time every interval_s while waiting."""
    t0 = time.time()
    while not stop_evt.wait(interval_s):
        print(f'    [still loading {file_name} ... {fmt_s(time.time()-t0)} elapsed]',
              flush=True)

# ──────────────────────────────────────────────────────────────────────────────

def load_and_merge_folder(folder, epoch_label, global_shift, subsample_n,
                          timeout_s=FILE_LOAD_TIMEOUT_S):
    """
    Load every LAS file in *folder*, subsample each to *subsample_n* points,
    then merge into one cloud with cc.MergeEntities.
    Files that hang longer than *timeout_s* are skipped.
    Returns the merged cloud.
    """
    folder = Path(folder)
    files  = sorted(folder.glob('*.las*'))
    if not files:
        raise RuntimeError(f'No .las files found in {folder}')

    n_files = len(files)
    timeout_msg = f'{timeout_s}s timeout' if timeout_s else 'no timeout'
    print(f'[LOAD] {epoch_label}: {n_files} files in {folder} ({timeout_msg})', flush=True)

    clouds    = []
    skipped   = []
    t0_total  = tnow()

    for i, f in enumerate(files, 1):
        t0 = tnow()
        p  = gunzip_if_needed(f)
        print(f'  [{i}/{n_files}] {p.name} — loading ...', flush=True)

        # Start heartbeat thread so hung files are visible
        stop_evt = threading.Event()
        hb = threading.Thread(target=_heartbeat, args=(p.name, stop_evt), daemon=True)
        hb.start()

        result = _load_with_timeout(str(p), global_shift, timeout_s)

        stop_evt.set()     # stop heartbeat
        hb.join(timeout=1)

        elapsed = fmt_s(tnow() - t0)

        if result == 'TIMEOUT':
            print(f'  [{i}/{n_files}] TIMEOUT ({timeout_s}s) — skipping {p.name}', flush=True)
            skipped.append(p.name)
            continue

        cloud = result
        if cloud is None:
            print(f'  [{i}/{n_files}] FAILED to load {p.name} — skipping.', flush=True)
            skipped.append(p.name)
            continue

        n_orig = int(cloud.size())
        if subsample_n and n_orig > subsample_n:
            sub = subsample_cloud(cloud, subsample_n)
            if sub is not cloud: safe_delete(cloud)
            cloud = sub

        print(f'  [{i}/{n_files}] OK  {p.name}: {n_orig:,} → {cloud.size():,} pts '
              f'| {elapsed} | mem={mem_gb():.2f}GB', flush=True)
        clouds.append(cloud)

    if skipped:
        print(f'[LOAD] {epoch_label}: skipped {len(skipped)} file(s): {skipped}', flush=True)

    if not clouds:
        raise RuntimeError(f'No clouds loaded from {folder}')

    print(f'[MERGE] {epoch_label}: merging {len(clouds)} clouds ...', flush=True)
    merged = cc.MergeEntities(clouds, deleteOriginalClouds=True)
    print(f'[MERGE] {epoch_label}: merged → {merged.size():,} pts '
          f'| total {fmt_s(tnow()-t0_total)} | mem={mem_gb():.2f}GB', flush=True)
    return merged

print('Load & merge functions ready.')

## 4. Overlap Detection & Clipping
Uses `cloud.toNpArrayCopy()` to get XYZ as a numpy array, then builds a 2D voxel occupancy grid at `OVERLAP_VOXEL_M` resolution. Cells present in **both** epochs define the shared coverage area. Both clouds are clipped to the bounding box of shared cells + buffer.

This catches irregular flight strip edges that a simple bounding-box intersection misses.

In [ ]:
def _crop_to_bbox(cloud, xmin, xmax, ymin, ymax, zmin, zmax):
    """
    Crop a cloud to an axis-aligned bounding box.
    Uses a temporary scalar field + filterPointsByScalarValue since
    cc.cropPointCloud is not available in all cloudComPy builds.
    """
    pts    = cloud.toNpArrayCopy()   # (N, 3) XYZ
    inside = (
        (pts[:, 0] >= xmin) & (pts[:, 0] <= xmax) &
        (pts[:, 1] >= ymin) & (pts[:, 1] <= ymax) &
        (pts[:, 2] >= zmin) & (pts[:, 2] <= zmax)
    ).astype(np.float32)

    sf_idx = cloud.addScalarField('_crop_mask')
    sf     = cloud.getScalarField(sf_idx)
    sf.fromNpArrayCopy(inside)
    sf.computeMinAndMax()
    cloud.setCurrentDisplayedScalarField(sf_idx)
    cropped = cloud.filterPointsByScalarValue(0.5, 1.5)
    cloud.deleteScalarField(sf_idx)
    return cropped


def clip_to_actual_overlap(c2017, c2018, voxel_m=OVERLAP_VOXEL_M,
                           buffer_m=OVERLAP_BUFFER_M, min_frac=MIN_OVERLAP_FRAC):
    """
    Find the area actually covered by BOTH epochs using a 2D voxel grid,
    then crop both clouds to that area + buffer.

    cloud.toNpArrayCopy() returns shape (N, 3) — columns are X, Y, Z
    in the shifted coordinate frame.
    """
    print(f'[OVERLAP] Computing 2D voxel occupancy at {voxel_m}m resolution ...', flush=True)

    def occupied_cells(cloud, vsize):
        pts = cloud.toNpArrayCopy()
        cx  = (pts[:, 0] / vsize).astype(np.int32)
        cy  = (pts[:, 1] / vsize).astype(np.int32)
        return set(zip(cx.tolist(), cy.tolist()))

    cells_17 = occupied_cells(c2017, voxel_m)
    cells_18 = occupied_cells(c2018, voxel_m)
    shared   = cells_17 & cells_18

    n_smaller = min(len(cells_17), len(cells_18))
    frac = len(shared) / max(n_smaller, 1)

    print(f'[OVERLAP] 2017 cells: {len(cells_17):,} | 2018 cells: {len(cells_18):,} '
          f'| shared: {len(shared):,} ({frac:.1%})', flush=True)

    if frac < min_frac:
        raise ValueError(
            f'[OVERLAP] Only {frac:.1%} shared coverage — '
            f'check that DIR_2017 and DIR_2018 cover the same area.')

    # Bounding box of shared cells
    shared_arr   = np.array(list(shared), dtype=np.float64)
    x_min_shared = float(shared_arr[:, 0].min()) * voxel_m
    x_max_shared = float(shared_arr[:, 0].max()) * voxel_m + voxel_m
    y_min_shared = float(shared_arr[:, 1].min()) * voxel_m
    y_max_shared = float(shared_arr[:, 1].max()) * voxel_m + voxel_m

    # Z: keep full range of both clouds
    pts_17 = c2017.toNpArrayCopy()
    pts_18 = c2018.toNpArrayCopy()
    z_min  = float(min(pts_17[:, 2].min(), pts_18[:, 2].min()))
    z_max  = float(max(pts_17[:, 2].max(), pts_18[:, 2].max()))
    del pts_17, pts_18

    xmin = x_min_shared - buffer_m;  xmax = x_max_shared + buffer_m
    ymin = y_min_shared - buffer_m;  ymax = y_max_shared + buffer_m
    zmin = z_min - 1.0;              zmax = z_max + 1.0

    print(f'[OVERLAP] Clipping to shared bbox + {buffer_m}m buffer ...', flush=True)
    c17_clipped = _crop_to_bbox(c2017, xmin, xmax, ymin, ymax, zmin, zmax)
    c18_clipped = _crop_to_bbox(c2018, xmin, xmax, ymin, ymax, zmin, zmax)

    for label, orig, cropped in [('2017', c2017, c17_clipped), ('2018', c2018, c18_clipped)]:
        if cropped is None or cropped.size() == 0:
            raise ValueError(f'[OVERLAP] {label} cloud is empty after clipping — check coordinates.')
        print(f'[OVERLAP] {label}: {orig.size():,} → {cropped.size():,} pts after clip', flush=True)

    overlap_report = {
        'voxel_size_m': voxel_m, 'buffer_m': buffer_m,
        'cells_2017': len(cells_17), 'cells_2018': len(cells_18),
        'shared_cells': len(shared), 'shared_fraction': frac,
        'clip_xmin': xmin, 'clip_xmax': xmax,
        'clip_ymin': ymin, 'clip_ymax': ymax,
        'pts_2017_before': int(c2017.size()), 'pts_2017_after': int(c17_clipped.size()),
        'pts_2018_before': int(c2018.size()), 'pts_2018_after': int(c18_clipped.size()),
    }
    return c17_clipped, c18_clipped, overlap_report

print('Overlap detection function ready.')

## 5. C2C Distance Computation

In [ ]:
def compute_c2c(src, tgt, sf_name, threads=12):
    t0 = tnow()
    print(f'[C2C] {sf_name}: approx pass ...', flush=True)
    approx = cc.DistanceComputationTools.computeApproxCloud2CloudDistance(src, tgt)
    if approx:
        print(f'[C2C] approx: min={approx[0]:.2f} max={approx[1]:.2f} mean={approx[2]:.2f}', flush=True)
    best_level = cc.DistanceComputationTools.determineBestOctreeLevel(src, None, tgt)
    print(f'[C2C] octree level = {best_level}', flush=True)
    params = cc.Cloud2CloudDistancesComputationParams()
    params.octreeLevel    = best_level
    params.multiThread    = True
    params.maxThreadCount = int(threads)
    print(f'[C2C] exact pass (threads={threads}) ...', flush=True)
    ret = cc.DistanceComputationTools.computeCloud2CloudDistances(src, tgt, params)
    if ret < 0: print(f'[C2C] WARNING: returned {ret}', flush=True)
    sf = src.getScalarField(src.getNumberOfScalarFields() - 1)
    sf.setName(sf_name); sf.computeMinAndMax()
    for name, idx in src.getScalarFieldDic().items():
        if 'approx' in name.lower():
            src.deleteScalarField(idx); break
    print(f'[C2C] done in {fmt_s(tnow()-t0)} | mem={mem_gb():.2f}GB', flush=True)
    return sf_name

def get_sf_as_array(cloud, sf_name):
    sf = cloud.getScalarField(sf_name)
    if sf is None:
        d = cloud.getScalarFieldDic()
        if sf_name in d: sf = cloud.getScalarField(d[sf_name])
    if sf is None:
        raise KeyError(f"SF '{sf_name}' not found. Available: {list(cloud.getScalarFieldDic())}")
    return np.asarray(sf.toNpArrayCopy(), dtype=np.float64)

print('C2C functions ready.')

## 6. Statistical Analysis & Plotting

In [ ]:
def c2c_summary_stats(c2c, label=''):
    clean = c2c[c2c <= NOISE_CEIL_M]; n_total = len(c2c); n_clean = len(clean)
    out = {'label': label, 'n_total': n_total, 'n_clean': n_clean,
           'n_noise': n_total-n_clean, 'pct_noise': (n_total-n_clean)/max(n_total,1)*100}
    if n_clean > 0:
        out.update({'mean': float(np.mean(clean)),   'median': float(np.median(clean)),
                    'std':  float(np.std(clean)),    'min':    float(np.min(clean)),
                    'max':  float(np.max(clean)),    'p05':    float(np.percentile(clean,5)),
                    'p25':  float(np.percentile(clean,25)), 'p75': float(np.percentile(clean,75)),
                    'p95':  float(np.percentile(clean,95)), 'p99': float(np.percentile(clean,99))})
    return out

def c2c_threshold_fractions(c2c):
    clean = c2c[c2c <= NOISE_CEIL_M]; n = max(len(clean), 1)
    return {t: float(np.sum(clean >= t)/n) for t in C2C_THRESHOLDS}

def extreme_value_report(c2c, label=''):
    """Report points above EXTREME_THRESHOLD_M — fixed physical threshold, not percentile."""
    clean   = c2c[c2c <= NOISE_CEIL_M]
    extreme = c2c[c2c >= EXTREME_THRESHOLD_M]   # uses raw array, above noise ceiling too
    if len(clean) == 0: return {'label': label, 'error': 'no clean points'}
    return {
        'label': label,
        'extreme_threshold_m': EXTREME_THRESHOLD_M,
        'n_extreme': len(extreme),
        'n_total':   len(c2c),
        'pct_extreme': float(len(extreme) / max(len(c2c), 1) * 100),
        'extreme_mean': float(np.mean(extreme)) if len(extreme) else float('nan'),
        'extreme_max':  float(np.max(extreme))  if len(extreme) else float('nan'),
        'pct_25_50m':  float(np.sum((extreme>=25) &(extreme< 50))/max(len(extreme),1)*100) if len(extreme) else 0,
        'pct_50_100m': float(np.sum((extreme>=50) &(extreme<100))/max(len(extreme),1)*100) if len(extreme) else 0,
        'pct_100m_up': float(np.sum( extreme>=100               )/max(len(extreme),1)*100) if len(extreme) else 0,
    }

def plot_c2c(c2c_arr, out_dir=PLOT_DIR):
    clean = c2c_arr[c2c_arr <= NOISE_CEIL_M]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Full-Area C2C Gap Analysis — 2017→2018 (Hurricane Maria)', fontsize=14, fontweight='bold')

    ax=axes[0,0]; ax.hist(clean, bins=np.linspace(0,HIST_MAX_M,HIST_BINS+1), density=True, color='red', alpha=0.7)
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('Density'); ax.set_title('(a) C2C Distribution'); ax.set_xlim(0,HIST_MAX_M)

    ax=axes[0,1]; s=np.sort(clean); step=max(len(s)//10000,1)
    ax.plot(s[::step], np.arange(1,len(s)+1)[::step]/len(s), color='red', lw=1.5)
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('CDF'); ax.set_title('(b) CDF'); ax.set_xlim(0,HIST_MAX_M); ax.grid(True,alpha=0.3)

    ax=axes[0,2]; tf=c2c_threshold_fractions(c2c_arr); x_pos=np.arange(len(C2C_THRESHOLDS))
    ax.bar(x_pos, [tf[t]*100 for t in C2C_THRESHOLDS], color='red', alpha=0.7)
    ax.set_xticks(x_pos); ax.set_xticklabels([f'>={t}m' for t in C2C_THRESHOLDS])
    ax.set_ylabel('% clean pts'); ax.set_title('(c) Threshold Exceedance')

    # (d) Fixed extreme outlier bands (>= EXTREME_THRESHOLD_M)
    ax=axes[1,0]
    ev = extreme_value_report(c2c_arr)
    bands      = ['25-50m', '50-100m', '>=100m']
    band_vals  = [ev.get('pct_25_50m',0), ev.get('pct_50_100m',0), ev.get('pct_100m_up',0)]
    ax.bar(np.arange(3), band_vals, color='red', alpha=0.7)
    ax.set_xticks(np.arange(3)); ax.set_xticklabels(bands)
    ax.set_ylabel(f'% of extreme pts (>={EXTREME_THRESHOLD_M}m)')
    ax.set_title(f'(d) Extreme Outliers (>={EXTREME_THRESHOLD_M}m)  n={ev["n_extreme"]:,}  ({ev["pct_extreme"]:.2f}% of all pts)')

    ax=axes[1,1]
    ax.hist(clean, bins=np.linspace(0,5,100), density=True, color='red', alpha=0.7)
    ax.axvline(2.0, color='gray', linestyle='--', alpha=0.7, label='2m threshold')
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('Density'); ax.set_title('(e) Detail 0–5m'); ax.legend(fontsize=8)

    ax=axes[1,2]; ax.axis('off')
    s1 = c2c_summary_stats(c2c_arr)
    txt = (f"Full Area — 2017→2018\n"
           f"  n={s1['n_clean']:,}  noise={s1['n_noise']:,} ({s1['pct_noise']:.1f}%)\n"
           f"  mean={s1.get('mean',0):.2f}m  median={s1.get('median',0):.2f}m\n"
           f"  std={s1.get('std',0):.2f}  p95={s1.get('p95',0):.2f}  p99={s1.get('p99',0):.2f}\n"
           f"  threshold fractions:\n"
           + '\n'.join(f"    >= {t}m : {tf[t]*100:.1f}%" for t in C2C_THRESHOLDS)
           + f"\n\n  extreme (>={EXTREME_THRESHOLD_M}m): {ev['n_extreme']:,} pts ({ev['pct_extreme']:.2f}%)"
           + f"\n  extreme mean={ev.get('extreme_mean',float('nan')):.1f}m  max={ev.get('extreme_max',float('nan')):.1f}m"
           + f"\n\nNOISE CEILING: {NOISE_CEIL_M}m")
    ax.text(0.05,0.95,txt,transform=ax.transAxes,fontsize=9,va='top',fontfamily='monospace',
            bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.5))
    ax.set_title('(f) Summary')
    plt.tight_layout(rect=[0,0,1,0.95])
    out = Path(out_dir)/'fullarea_c2c_analysis.png'
    fig.savefig(str(out), dpi=150); plt.close(fig)
    print(f'[PLOT] saved {out}', flush=True)

print('Statistical analysis and plotting functions ready.')

## 7. Connected Components & Power Law

In [ ]:
def _cfg(config, key): return config[key] if config and key in config else CC_CONFIG[key]

def _dynamic_octree_level(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    diag=sqrt((hi[0]-lo[0])**2+(hi[1]-lo[1])**2+(hi[2]-lo[2])**2)
    return max(6, min(12, int(log2(max(diag/2.0, 1.0))))) if diag>0 else 8

def extract_damage_components(source_cloud, c2c_array, sf_name, config=None):
    threshold = _cfg(config, 'damage_threshold')
    min_pts   = _cfg(config, 'min_component_pts')
    max_comps = _cfg(config, 'max_components')
    n_above   = int(np.sum(c2c_array > threshold))
    if n_above == 0:
        print(f'[COMP] No points exceed {threshold}m — nothing to extract.'); return [], [], 0
    print(f'[COMP] {n_above:,}/{c2c_array.size:,} pts ({100.0*n_above/c2c_array.size:.1f}%) above {threshold}m.', flush=True)
    sf_dic = source_cloud.getScalarFieldDic()
    if sf_name not in sf_dic:
        print(f'[COMP] WARNING — SF {sf_name!r} not found on cloud.'); return [], [], 0
    source_cloud.setCurrentDisplayedScalarField(sf_dic[sf_name])
    damaged = source_cloud.filterPointsByScalarValue(threshold, float(np.max(c2c_array))+1.0)
    if damaged is None or damaged.size() == 0:
        print('[COMP] WARNING — filter returned empty cloud.'); return [], [], 0
    print(f'[COMP] Damaged cloud: {damaged.size():,} pts', flush=True)
    level = _dynamic_octree_level(damaged)
    print(f'[COMP] Octree level = {level}', flush=True)
    _, components, residuals = cc.ExtractConnectedComponents(
        clouds=[damaged], octreeLevel=level,
        minComponentSize=min_pts, maxNumberComponents=max_comps, randomColors=False)
    print(f'[COMP] {len(components):,} components | {len(residuals)} residual clouds', flush=True)
    cc.deleteEntity(damaged)
    return components, residuals, level

def compute_component_stats(components, residuals, point_density=None):
    stats = []
    for i, comp in enumerate(components):
        bb=comp.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
        bbox_area = (hi[0]-lo[0])*(hi[1]-lo[1])
        sf_dict = comp.getScalarFieldDic()
        if sf_dict:
            arr = comp.getScalarField(next(iter(sf_dict))).toNpArrayCopy()
            mean_c2c=float(np.mean(arr)); max_c2c=float(np.max(arr))
        else:
            mean_c2c=max_c2c=float('nan')
        rec = {'component_id':i,'n_points':int(comp.size()),
               'bbox_area_m2':bbox_area,'bbox_area_sqrt_m':sqrt(max(bbox_area,0)),
               'mean_c2c_m':mean_c2c,'max_c2c_m':max_c2c,
               'centroid_x':0.5*(lo[0]+hi[0]),'centroid_y':0.5*(lo[1]+hi[1])}
        if point_density and point_density > 0:
            rec['estimated_area_m2'] = int(comp.size()) / point_density
        stats.append(rec)
    print(f'[COMP] Stats computed for {len(stats):,} components.', flush=True)
    return stats

def _delete_cloud_list(clouds):
    for c in clouds:
        try: cc.deleteEntity(c)
        except: pass
    clouds.clear()

print('Connected component functions ready.')

In [ ]:
def fit_power_law(areas, label='', min_area_m2=10.0):
    areas  = np.asarray(areas, dtype=np.float64)
    fitted = areas[areas >= min_area_m2]
    result = {'label':label,'alpha_mle':float('nan'),'alpha_ols':float('nan'),
               'xmin':min_area_m2,'n_fitted':0,'ks_stat':float('nan'),'ks_pvalue':float('nan'),
               'r2_loglog':float('nan'),'ll_powerlaw':float('nan'),'ll_lognormal':float('nan'),
               'preferred_distribution':'undetermined'}
    if fitted.size < 5:
        print(f'[POWERLAW] {label}: only {fitted.size} areas >= {min_area_m2}m² — too few.'); return result
    n=fitted.size; xmin=min_area_m2; result['n_fitted']=n
    sum_log=float(np.sum(np.log(fitted/xmin)))
    alpha_mle=(1.0+n/sum_log) if sum_log>0 else float('nan')
    result['alpha_mle']=alpha_mle
    n_bins=max(10,int(sqrt(n))); log_edges=np.linspace(np.log10(xmin),np.log10(fitted.max()),n_bins+1)
    counts,_=np.histogram(np.log10(fitted),bins=log_edges); pos=counts>0
    if pos.sum()>=2:
        slope,_,r,_,_=sp_stats.linregress(0.5*(log_edges[:-1]+log_edges[1:])[pos],np.log10(counts[pos].astype(float)))
        result['alpha_ols']=-slope; result['r2_loglog']=r**2
    if np.isfinite(alpha_mle) and alpha_mle>1.0:
        ks=sp_stats.kstest(fitted,'pareto',args=(alpha_mle-1.0,),N=n)
        result['ks_stat']=float(ks.statistic); result['ks_pvalue']=float(ks.pvalue)
        ll_pl=float(n*np.log(alpha_mle-1.0)-n*np.log(xmin)-alpha_mle*np.sum(np.log(fitted/xmin)))
        result['ll_powerlaw']=ll_pl
    else: ll_pl=-np.inf
    mu_ln=float(np.mean(np.log(fitted))); sig_ln=float(np.std(np.log(fitted),ddof=1)) if n>1 else 1.0
    ll_ln=float(np.sum(sp_stats.lognorm.logpdf(fitted,s=sig_ln,scale=np.exp(mu_ln)))) if sig_ln>0 else -np.inf
    result['ll_lognormal']=ll_ln
    if np.isfinite(ll_pl) and np.isfinite(ll_ln):
        result['preferred_distribution']='power_law' if ll_pl>ll_ln else 'lognormal'
    print(f"[POWERLAW] {label}: α_MLE={alpha_mle:.3f} α_OLS={result['alpha_ols']:.3f} "
          f"n={n} KS_p={result['ks_pvalue']:.4f} preferred={result['preferred_distribution']}")
    return result

def gap_size_figure(comp_stats, pl_fit, out_dir=PLOT_DIR):
    areas    = np.array([s['bbox_area_m2'] for s in comp_stats], dtype=np.float64)
    mean_c2c = np.array([s['mean_c2c_m']  for s in comp_stats], dtype=np.float64)
    if areas.size == 0: return
    pos = areas[areas>0]
    fig, axes = plt.subplots(2, 2, figsize=(12,10))
    fig.suptitle('Full-Area Gap Size Distribution — 2017→2018', fontsize=13)
    ax=axes[0,0]
    if pos.size > 0:
        log_bins=np.logspace(np.log10(pos.min()),np.log10(pos.max()),max(10,int(sqrt(pos.size))))
        ax.hist(pos,bins=log_bins,edgecolor='k',alpha=0.7,color='red',label='data')
        alpha=pl_fit['alpha_mle']; xmin=pl_fit['xmin']
        if np.isfinite(alpha) and alpha>1.0:
            xl=np.logspace(np.log10(xmin),np.log10(pos.max()),200)
            ax.plot(xl,((alpha-1.0)/xmin)*(xl/xmin)**(-alpha)*np.sum(pos>=xmin)*float(np.mean(np.diff(log_bins))),
                    'k-',lw=2,label=f'PL α={alpha:.2f}')
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Gap area (m²)'); ax.set_ylabel('Count')
    ax.set_title('(a) Log-log histogram'); ax.legend(fontsize=8)
    r2=pl_fit.get('r2_loglog',float('nan')); n=pl_fit['n_fitted']
    ax.text(0.97,0.97,f"α={pl_fit['alpha_mle']:.2f}\nR²={r2:.3f}\nn={n:,}",
            transform=ax.transAxes,fontsize=8,va='top',ha='right',bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))
    sv=np.sort(areas); ax=axes[0,1]
    ax.step(sv,np.arange(1,len(sv)+1)/len(sv),where='post',color='red',lw=1.5,label='Empirical')
    alpha=pl_fit['alpha_mle']; xmin=pl_fit['xmin']
    if np.isfinite(alpha) and alpha>1.0:
        x_th=np.sort(sv[sv>=xmin])
        if x_th.size>0:
            fb=np.searchsorted(sv,xmin)/len(sv)
            ax.plot(x_th,fb+(1-fb)*(1-(x_th/xmin)**(-(alpha-1))),'k--',lw=1.5,label='PL CDF')
    ax.set_xscale('log'); ax.set_xlabel('Gap area (m²)'); ax.set_ylabel('CDF')
    ax.set_title('(b) Empirical vs PL CDF'); ax.legend(fontsize=8)
    ax=axes[1,0]; valid=np.isfinite(mean_c2c)&(areas>0)
    if valid.any(): ax.scatter(areas[valid],mean_c2c[valid],s=4,alpha=0.3,color='red')
    ax.set_xscale('log'); ax.set_xlabel('Gap area (m²)'); ax.set_ylabel('Mean C2C (m)')
    ax.set_title('(c) Gap area vs C2C displacement')
    ax=axes[1,1]
    ax.scatter(np.arange(1,len(sv)+1),sv[::-1],s=4,alpha=0.4,color='red')
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Rank'); ax.set_ylabel('Gap area (m²)')
    ax.set_title('(d) Rank-size (Zipf)')
    fig.tight_layout(rect=[0,0,1,0.95])
    out=Path(out_dir)/'fullarea_gap_distribution.png'
    fig.savefig(out,dpi=150); plt.close(fig)
    print(f'[DIST] Saved {out}', flush=True)

print('Power law and distribution functions ready.')

## 8. Determine Global Coordinate Shift
Load one file to read CloudCompare's auto-shift, then force that same shift on every file.

In [ ]:
probe_file = next(Path(DIR_2017).glob('*.las*'), None)
if probe_file is None:
    raise RuntimeError(f'No .las files found in {DIR_2017}')

print(f'[SHIFT] Probing {probe_file.name} ...', flush=True)
probe = cc.loadPointCloud(str(gunzip_if_needed(probe_file)))
if probe is None:
    raise RuntimeError('Failed to load probe file')

global_shift = tuple(float(x) for x in probe.getGlobalShift()) if probe.isShifted() else (0.0,0.0,0.0)
bb = probe.getOwnBB()
print(f'[SHIFT] shift = {global_shift}')
print(f'[SHIFT] BB: X=[{float(bb.minCorner()[0]):.1f},{float(bb.maxCorner()[0]):.1f}] '
      f'Y=[{float(bb.minCorner()[1]):.1f},{float(bb.maxCorner()[1]):.1f}] '
      f'Z=[{float(bb.minCorner()[2]):.1f},{float(bb.maxCorner()[2]):.1f}]')
safe_delete(probe)
print(f'\nUsing shift {global_shift} for ALL files.')

## 9. Run Pipeline
Four steps — run them in order. Each step saves its result so you can re-run individual steps without redoing everything.

In [ ]:
# ── Step 1: Load & merge both folders ─────────────────────────────────────────
print('STEP 1: Loading and merging folders\n' + '='*60)
merged_2017 = load_and_merge_folder(DIR_2017, '2017', global_shift, SUBSAMPLE_N_PER_FILE)
merged_2018 = load_and_merge_folder(DIR_2018, '2018', global_shift, SUBSAMPLE_N_PER_FILE)
print(f'\nMerged 2017: {merged_2017.size():,} pts | Merged 2018: {merged_2018.size():,} pts')

In [ ]:
# ── Step 2: Clip to actual overlap ────────────────────────────────────────────
print('STEP 2: Finding overlap and clipping\n' + '='*60)
c2017_clipped, c2018_clipped, overlap_report = clip_to_actual_overlap(merged_2017, merged_2018)

# Free the unclipped merged clouds — no longer needed
safe_delete(merged_2017); safe_delete(merged_2018)

print(f'\nOverlap: {overlap_report["shared_fraction"]:.1%} shared coverage')
print(f'Clipped 2017: {c2017_clipped.size():,} pts')
print(f'Clipped 2018: {c2018_clipped.size():,} pts')
print(f'mem={mem_gb():.2f}GB')

In [ ]:
# ── Step 3: C2C ───────────────────────────────────────────────────────────────
print('STEP 3: Computing C2C distances\n' + '='*60)
sf_name = compute_c2c(c2017_clipped, c2018_clipped, 'C2C_2017_2018', threads=THREADS)
c2c_arr = get_sf_as_array(c2017_clipped, sf_name)

# Save the 2017 cloud with C2C scalar field attached — can reload later
cc.SavePointCloud(c2017_clipped, str(BIN_DIR / 'fullarea_2017_c2c.bin'))
print(f'C2C array: {len(c2c_arr):,} pts | mean={np.mean(c2c_arr[c2c_arr<=NOISE_CEIL_M]):.2f}m | mem={mem_gb():.2f}GB')

# C2C diagnostic plot
plot_c2c(c2c_arr)

# 2018 cloud no longer needed after C2C
safe_delete(c2018_clipped)

In [ ]:
# ── Step 4: Connected components + power law ──────────────────────────────────
print('STEP 4: Connected components and gap size analysis\n' + '='*60)

# Point density for area estimation
bb   = c2017_clipped.getOwnBB()
xy_area = (bb.maxCorner()[0]-bb.minCorner()[0]) * (bb.maxCorner()[1]-bb.minCorner()[1])
density = c2017_clipped.size() / max(float(xy_area), 1.0)
print(f'[COMP] Point density = {density:.2f} pts/m²', flush=True)

components, residuals, octree_level = extract_damage_components(
    c2017_clipped, c2c_arr, sf_name, CC_CONFIG)

comp_stats = compute_component_stats(components, residuals, density)

areas = np.array([s['bbox_area_m2'] for s in comp_stats], dtype=np.float64)
pl_fit = fit_power_law(areas, label='full_area')
gap_size_figure(comp_stats, pl_fit)

_delete_cloud_list(components)
_delete_cloud_list(residuals)
safe_delete(c2017_clipped)

print(f'\nStep 4 complete | mem={mem_gb():.2f}GB')

## 10. Write Outputs

In [ ]:
write_csv(CSV_DIR/'c2c_summary.csv',       [c2c_summary_stats(c2c_arr, 'full_area_2017_2018')])
write_csv(CSV_DIR/'c2c_thresholds.csv',    [{**{'label':'full_area'}, **c2c_threshold_fractions(c2c_arr)}])
write_csv(CSV_DIR/'c2c_extreme.csv',       [extreme_value_report(c2c_arr, 'full_area')])
write_csv(CSV_DIR/'component_stats.csv',   comp_stats)
write_csv(CSV_DIR/'power_law_fit.csv',     [{k:v for k,v in pl_fit.items() if k!='areas_fitted'}])
write_csv(CSV_DIR/'overlap_report.csv',    [overlap_report])
print('All outputs written to', CSV_DIR)

## 11. Results Review

In [ ]:
import pandas as pd

print('=== C2C Summary ===')
print(pd.read_csv(CSV_DIR/'c2c_summary.csv').T.to_string(header=False))

print('\n=== Power Law Fit ===')
print(pd.read_csv(CSV_DIR/'power_law_fit.csv').T.to_string(header=False))

print('\n=== Overlap Report ===')
print(pd.read_csv(CSV_DIR/'overlap_report.csv').T.to_string(header=False))

print('\n=== Component Stats (top 10 by area) ===')
df = pd.read_csv(CSV_DIR/'component_stats.csv')
print(df.nlargest(10,'bbox_area_m2')[['component_id','n_points','bbox_area_m2','mean_c2c_m','centroid_x','centroid_y']].to_string(index=False))
print(f'\nTotal components: {len(df):,}')
print(f'Total damaged area: {df.bbox_area_m2.sum():.0f} m²  ({df.bbox_area_m2.sum()/10000:.2f} ha)')